In [1]:
import pandas as pd
import numpy as np

In [6]:
attendance_data=pd.read_csv('../data/raw_data/attendance.csv')
homework_data=pd.read_csv('../data/raw_data/homework.csv')
perfomance_data=pd.read_csv('../data/raw_data/performance.csv') 
students_data=pd.read_csv('../data/raw_data/students.csv')  
teacher_parent_communication_data=pd.read_csv('../data/raw_data/teacher_parent_communication.csv')  

In [7]:
attendance_data

,Student_ID,Date,Subject,Attendance_Status
0,S06592,2024-09-20,Arabic,Present
1,S01777,2025-01-22,Math,Present
2,S07362,2024-12-03,Geography,PRESENT
3,S12119,2024-03-16,Geography,excused
4,S02002,2024-05-24,Math,PRESENT
...,...,...,...,...
364675,S08595,2025-02-14,Math,Present
364676,S11222,2025-03-06,Science,left early
364677,S11344,2024-04-18,History,Present
364678,S04381,2024-10-04,Science,PRESENT


In [8]:
attendance_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 364680 entries, 0 to 364679
Data columns (total 4 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   Student_ID         364680 non-null  str  
 1   Date               364680 non-null  str  
 2   Subject            364680 non-null  str  
 3   Attendance_Status  364680 non-null  str  
dtypes: str(4)
memory usage: 11.1 MB


In [9]:
attendance_data.duplicated().sum()

np.int64(290)

In [11]:
attendance_data[attendance_data.duplicated()].head(10)

,Student_ID,Date,Subject,Attendance_Status
32867,S08155,2024-08-12,Geography,Absent
42044,S05539,2024-09-20,English,left early
44299,S05683,2024-11-02,Geography,absnt
51295,S04926,2024-08-18,Arabic,late
51419,S07223,2024-11-19,Arabic,Present
52233,S02043,2024-03-28,Science,PRESENT
52482,S08765,2025-03-06,History,absnt
52483,S02289,2024-04-30,Science,late
55812,S06640,2024-08-26,History,absnt
64892,S11843,2024-05-19,Science,Present


In [14]:
attendance_data.duplicated(subset=["Student_ID", "Date","Subject"]).sum()

np.int64(2380)

In [15]:
#the above code shows that there are 2380 duplicates when considered both id, date and subject, which is normally impossible, because a student can attend one subject once at the same day while the whole dataset duplicates showed 290 rows. maybe there are some inconsistencies in attendance_status column, let me first investigate it.

In [16]:
conflicts = (
    attendance_data.groupby(["Student_ID", "Date", "Subject"])["Attendance_Status"]
    .nunique()
)

conflicts = conflicts[conflicts > 1]
conflicts.count()

np.int64(2085)

In [ ]:
attendance_data[
    attendance_data.set_index(["Student_ID", "Date", "Subject"]).index.isin(conflicts.index)
].sort_values(["Student_ID", "Date", "Subject"])
#checking the exact values of the attendance status for duplicates to see the root course of the conflicts.

,Student_ID,Date,Subject,Attendance_Status
245552,S00004,2024-12-24,Geography,absnt
302174,S00004,2024-12-24,Geography,Late
21914,S00007,2024-10-14,English,excused
190486,S00007,2024-10-14,English,absnt
285963,S00007,2025-03-04,Arabic,Present
...,...,...,...,...
286924,S12145,2024-07-11,English,Absent
82607,S12146,2024-11-22,Arabic,absnt
304478,S12146,2024-11-22,Arabic,Absent
64478,S12152,2024-07-29,English,excused


In [23]:
#resolving typo-errors in attendance status column.
attendance_data["Attendance_Status"] = attendance_data["Attendance_Status"].str.lower().str.strip()

attendance_data["Attendance_Status"] = attendance_data["Attendance_Status"].replace({
    "absnt": "Absent",
    "absent": "Absent",
    "present": "Present",
    "late": "Late",
    "left early": "Left Early",
    "excused": "Excused"
})

In [25]:
attendance_data["Attendance_Status"].unique()
#  i can see that the typo erros are solved and now i have the well formated texts

<StringArray>
['Present', 'Excused', 'Late', 'Absent', 'Left Early']
Length: 5, dtype: str

In [19]:
attendance_data[
    attendance_data.set_index(["Student_ID", "Date", "Subject"]).index.isin(conflicts.index)
].sort_values(["Student_ID", "Date", "Subject"])

,Student_ID,Date,Subject,Attendance_Status
245552,S00004,2024-12-24,Geography,Absent
302174,S00004,2024-12-24,Geography,Late
21914,S00007,2024-10-14,English,Excused
190486,S00007,2024-10-14,English,Absent
285963,S00007,2025-03-04,Arabic,Present
...,...,...,...,...
286924,S12145,2024-07-11,English,Absent
82607,S12146,2024-11-22,Arabic,Absent
304478,S12146,2024-11-22,Arabic,Absent
64478,S12152,2024-07-29,English,Excused


In [21]:
attendance_data.groupby(
    ["Student_ID", "Date", "Subject"]
)["Attendance_Status"].nunique().gt(1).sum()

np.int64(1862)

In [22]:
attendance_data.duplicated().sum()

np.int64(515)

In [28]:
#now i am removing all duplicates 515 duplicates.
attendance_data = attendance_data.drop_duplicates()
attendance_data.duplicated().sum()

np.int64(0)

In [29]:
attendance_data.duplicated(
    subset=["Student_ID", "Date", "Subject"]
).sum()

np.int64(1865)

In [30]:
#as now the duplicates removed but still having conflicts in attendance status, i will resolve this based on priority of the attendance status as follow: 
# Absent → student did not attend(5)

#Excused → absent but with valid reason(4)

#Left Early → partial attendance(3)

#Late → partial attendance(2)

#Present → full attendance(1)
#Higher number = stronger absence signal.

In [36]:
priority = {
    "Absent": 5,
    "Excused": 4,
    "Left Early": 3,
    "Late": 2,
    "Present": 1
}

attendance_data["priority"] = attendance_data["Attendance_Status"].map(priority)

attendance_clean = (
    attendance_data
    .sort_values("priority", ascending=False)
    .drop_duplicates(
        subset=["Student_ID", "Date", "Subject"]
    )
    .drop(columns="priority")
)

In [38]:
attendance_clean.duplicated(
    subset=["Student_ID","Date","Subject"]
).sum()

np.int64(0)

In [35]:
attendance_clean

,Student_ID,Date,Subject,Attendance_Status
8,S09604,2024-08-06,Arabic,Absent
272925,S10319,2024-10-29,Math,Absent
272928,S07741,2024-07-12,Science,Absent
272931,S05288,2024-08-27,Math,Absent
272932,S03050,2024-11-20,English,Absent
...,...,...,...,...
272946,S04492,2024-05-01,History,Present
272945,S07061,2024-05-31,Science,Present
272943,S03778,2024-10-09,Science,Present
272942,S10274,2024-04-22,Science,Present


In [40]:
attendance_clean["Attendance_Status"].unique()

<StringArray>
['Absent', 'Excused', 'Left Early', 'Late', 'Present']
Length: 5, dtype: str

In [43]:
attendance_clean["priority"] = attendance_clean["Attendance_Status"].map(priority)
attendance_clean

,Student_ID,Date,Subject,Attendance_Status,priority
8,S09604,2024-08-06,Arabic,Absent,5
272925,S10319,2024-10-29,Math,Absent,5
272928,S07741,2024-07-12,Science,Absent,5
272931,S05288,2024-08-27,Math,Absent,5
272932,S03050,2024-11-20,English,Absent,5
...,...,...,...,...,...
272946,S04492,2024-05-01,History,Present,1
272945,S07061,2024-05-31,Science,Present,1
272943,S03778,2024-10-09,Science,Present,1
272942,S10274,2024-04-22,Science,Present,1


In [45]:
attendance_clean["priority"].isna().sum()

np.int64(0)

In [46]:
attendance_clean["Date"].dtype

<StringDtype(storage='python', na_value=nan)>

In [48]:
# as the date columns are in text format, i need to convert it to datetime format.
attendance_clean["Date"] = pd.to_datetime(
    attendance_clean["Date"],
    errors="coerce"
)

In [49]:
attendance_clean["Date"].dtype

dtype('<M8[us]')

In [50]:
attendance_clean["Date"].isna().sum()

np.int64(0)

In [52]:
attendance_clean["Date"].min()

Timestamp('2024-03-09 00:00:00')

In [53]:
attendance_clean["Date"].max()

Timestamp('2025-03-09 00:00:00')

In [54]:
attendance_clean

,Student_ID,Date,Subject,Attendance_Status,priority
8,S09604,2024-08-06,Arabic,Absent,5
272925,S10319,2024-10-29,Math,Absent,5
272928,S07741,2024-07-12,Science,Absent,5
272931,S05288,2024-08-27,Math,Absent,5
272932,S03050,2024-11-20,English,Absent,5
...,...,...,...,...,...
272946,S04492,2024-05-01,History,Present,1
272945,S07061,2024-05-31,Science,Present,1
272943,S03778,2024-10-09,Science,Present,1
272942,S10274,2024-04-22,Science,Present,1


In [55]:
attendance_clean.isna().sum()

Student_ID           0
Date                 0
Subject              0
Attendance_Status    0
priority             0
dtype: int64

In [58]:
attendance_clean["Attendance_Status"].value_counts(normalize=True)*100

Attendance_Status
Absent        25.134143
Present       24.990616
Late          24.912227
Excused       12.482197
Left Early    12.480817
Name: proportion, dtype: float64

In [59]:
attendance_clean["Attendance_Status"] = pd.Categorical(
    attendance_clean["Attendance_Status"],
    categories=["Present", "Late", "Left Early", "Excused", "Absent"],
    ordered=True
)

In [61]:
attendance_clean["Subject"].unique()

<StringArray>
['Arabic', 'Math', 'Science', 'English', 'History', 'Geography']
Length: 6, dtype: str

In [62]:
attendance_clean["Subject"].value_counts()

Subject
Math         60830
Science      60457
Arabic       60337
English      60310
Geography    60261
History      60105
Name: count, dtype: int64

In [64]:
attendance_clean.shape

(362300, 5)